In [ ]:
%load_ext tensorboard

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATASET_FILE = 'jetbot_dataset_2025-12-12_13-55-32.zip'  # <-- change if needed
DATASET_DIR = 'dataset'
DATASET_ZIP = os.path.join(DATASET_DIR, DATASET_FILE)

!ls /content/drive/MyDrive/dataset/

In [ ]:
!rm -rf dataset_root
!cp '/content/drive/MyDrive/{DATASET_ZIP}' ./
!unzip -q {DATASET_FILE}
!mkdir dataset_root
!mv {DATASET_DIR}/* './dataset_root'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets
import torchvision.transforms.v2 as transforms  # Updated to v2 to avoid deprecations
from torchvision.utils import save_image
from torch.utils.tensorboard import SummaryWriter  # For TensorBoard logging
from IPython.display import Image, display

# For perceptual loss
from torchvision.models import vgg16, VGG16_Weights

# For LR scheduler
from torch.optim.lr_scheduler import ReduceLROnPlateau

# For image validation
from PIL import Image

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
bs = 256  # Increased for A100 efficiency

# Augmentations for better generalization
transform = transforms.Compose([
    transforms.Resize((120, 160)),
    transforms.CenterCrop((80, 160)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

dataset = datasets.ImageFolder(
    root='./dataset_root',
    transform=transform
)

# Validate images
valid_count = 0
invalid_paths = []
for path, _ in dataset.samples:
    try:
        with Image.open(path) as img:
            img.verify()
        valid_count += 1
    except (IOError, SyntaxError) as e:
        invalid_paths.append(path)
        print(f"Invalid image skipped: {path} ({str(e)})")

print(f"Total images found: {len(dataset.samples)}")
print(f"Valid images: {valid_count}")
if invalid_paths:
    print(f"Invalid images: {len(invalid_paths)}")

# Filter dataset
dataset.samples = [sample for sample in dataset.samples if sample[0] not in invalid_paths]
dataset.imgs = dataset.samples

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=bs,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(f"Dataset length after filtering: {len(dataset)}")
print(f"Number of batches: {len(dataloader)}")

In [ ]:
class Flatten(nn.Module):
    def forward(self, input):
        return input.view(input.size(0), -1)

class UnFlatten(nn.Module):
    def forward(self, input, size=512):  # Updated to match moderate widening
        return input.view(input.size(0), size, 5, 10)

class VAE(nn.Module):
    def __init__(self, image_channels=3, z_dim=64):  # Moderate z_dim increase
        super(VAE, self).__init__()
        self.z_dim = z_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(image_channels, 32, 4, stride=2, padding=1),  # 3 x 80 x 160 -> 32 x 40 x 80
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # -> 64 x 20 x 40
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),  # -> 128 x 10 x 20
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),  # -> 256 x 5 x 10
            nn.ReLU(),
            nn.Conv2d(256, 512, 3, stride=1, padding=1),  # New mild layer: -> 512 x 5 x 10
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(512 * 5 * 10, z_dim)  # Updated input size
        self.fc_logvar = nn.Linear(512 * 5 * 10, z_dim)
        self.decoder_input = nn.Linear(z_dim, 512 * 5 * 10)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 3, stride=1, padding=1),  # Matches new encoder layer -> 256 x 5 x 10
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # -> 128 x 10 x 20
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # -> 64 x 20 x 40
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),  # -> 32 x 40 x 80
            nn.ReLU(),
            nn.ConvTranspose2d(32, image_channels, 4, stride=2, padding=1),  # -> 3 x 80 x 160
            nn.Tanh(),  # Outputs [-1,1]
        )
        # Perceptual loss VGG
        self.vgg = vgg16(weights=VGG16_Weights.DEFAULT).features[:16].eval().to(device)
        for param in self.vgg.parameters():
            param.requires_grad = False
        self.vgg_normalization = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                      std=[0.229, 0.224, 0.225])

    def encode(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        z = self.reparameterize(mu, logvar)
        return z, mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        x = self.decoder_input(z)
        x = x.view(x.size(0), 512, 5, 10)  # Updated channels
        x = self.decoder(x)
        return x

    def forward(self, x):
        z, mu, logvar = self.encode(x)
        recon = self.decode(z)
        return recon, mu, logvar

    def perceptual_loss(self, recon_x, x):
        x = (x + 1) / 2
        recon_x = (recon_x + 1) / 2
        x_norm = self.vgg_normalization(x)
        recon_x_norm = self.vgg_normalization(recon_x)
        feat_recon = self.vgg(recon_x_norm)
        feat_x = self.vgg(x_norm)
        return F.mse_loss(feat_recon, feat_x, reduction='mean')

    def loss_fn(self, image, recon, mean, logvar, beta=1.0, perc_weight=0.1):
        REC = F.mse_loss(recon, image, reduction='sum')
        KL = -0.5 * torch.sum(1 + logvar - mean.pow(2) - logvar.exp())
        p_loss = self.perceptual_loss(recon, image)
        PERC = p_loss * 10000  # Adjustable scaling
        total_loss = REC + beta * KL + perc_weight * PERC
        return total_loss, REC, beta * KL, perc_weight * PERC  # Return components for monitoring

In [ ]:
# Initialize model, optimizer, scheduler
vae = VAE().to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# TensorBoard writer
writer = SummaryWriter(log_dir='runs/vae_training')

In [ ]:
# Training loop with monitoring
num_epochs = 200  # Increased for better convergence
beta = 0.1  # Start low for annealing
perc_weight = 0.1

for epoch in range(num_epochs):
    vae.train()
    running_total = running_rec = running_kl = running_perc = 0.0
    for batch_idx, (data, _) in enumerate(dataloader):
        data = data.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = vae(data)
        total_loss, rec, kl, perc = vae.loss_fn(data, recon, mu, logvar, beta=beta, perc_weight=perc_weight)
        total_loss.backward()
        optimizer.step()
        
        running_total += total_loss.item()
        running_rec += rec.item()
        running_kl += kl.item()
        running_perc += perc.item()
        
        if batch_idx % 10 == 0:
            print(f"Epoch {epoch}, Batch {batch_idx}: Total={total_loss.item():.2f}, REC={rec.item():.2f}, KL={kl.item():.2f}, PERC={perc.item():.2f}")

    avg_total = running_total / len(dataloader)
    avg_rec = running_rec / len(dataloader)
    avg_kl = running_kl / len(dataloader)
    avg_perc = running_perc / len(dataloader)
    
    print(f"====> Epoch: {epoch} Average loss: {avg_total:.4f}")
    
    # Log to TensorBoard
    writer.add_scalar('Loss/Total', avg_total, epoch)
    writer.add_scalar('Loss/REC', avg_rec, epoch)
    writer.add_scalar('Loss/KL', avg_kl, epoch)
    writer.add_scalar('Loss/PERC', avg_perc, epoch)
    
    # Anneal beta
    beta = min(beta + 0.01, 1.0)  # Ramp to 1.0 over ~90 epochs
    
    # Scheduler step
    scheduler.step(avg_total)
    
    # Save checkpoint
    if epoch % 10 == 0:
        torch.save(vae.state_dict(), f'vae_epoch_{epoch}.torch')

writer.close()

In [ ]:
# For Jetson Nano deployment: Quantize model
import torch.quantization as quantization

vae.eval()
vae_quantized = quantization.quantize_dynamic(vae, {nn.Conv2d: quantization.default_dynamic_qconfig, nn.Linear: quantization.default_dynamic_qconfig}, dtype=torch.qint8)
torch.save(vae_quantized.state_dict(), 'vae_quantized.torch')

In [ ]:
%tensorboard --logdir runs